In [ ]:
# ======================================
# STEP 1: Install Dependencies
# ======================================
!pip install openai python-dotenv nest_asyncio


# ======================================
# STEP 2: Load API Key from .env
# ======================================
import os
from dotenv import load_dotenv
from openai import OpenAI
import asyncio
import random
import logging
import nest_asyncio

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)


# ======================================
# STEP 3: Simulated MCP Tools
# ======================================
async def page_security_team(payload):

    return "🚨 Security team paged"


async def create_jira_ticket(payload):

    return f"🎫 Jira ticket created: {payload}"


async def check_network_status(payload):

    return random.choice([

        "Network degraded",

        "All systems normal"
    ])


async def alert_noc_team(payload):

    return "📡 NOC team alerted"


async def get_ad_user(payload):

    return "user_123"


async def reset_permissions(payload):

    return "🔐 Permissions reset successfully"


async def query_postgres(payload):

    # simulate failure
    if random.random() < 0.7:

        raise Exception("Database timeout")

    return "📦 Data from Postgres"


async def query_sqlite_cache(payload):

    return "🗂 Data from SQLite cache (fallback)"


# ======================================
# STEP 4: LLM Classifier (NVIDIA)
# ======================================
def classify_issue(issue):

    prompt = f"""

    Classify IT issue into ONE:

    network
    hardware
    software
    security
    access

    Issue:
    {issue}

    Return only category name.
    """

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {

                "role": "user",

                "content": prompt
            }

        ]

    )

    return response.choices[0].message.content.strip().lower()


#======================================
# STEP 5: Conditional MCP Routing
# ======================================
async def smart_it_triage(issue, severity):

    issue_type = classify_issue(issue)

    print(f"🧠 Classified as: {issue_type}")


    if issue_type == "security":

        await page_security_team({"issue": issue})

        return await create_jira_ticket({

            "project": "SEC",

            "priority": "Blocker"
        })


    elif issue_type == "network" and severity == "high":

        status = await check_network_status({})

        if "degraded" in status.lower():

            return await alert_noc_team({

                "issue": issue
            })

        else:

            return await create_jira_ticket({

                "project": "NET",

                "priority": "High"
            })


    elif issue_type == "access":

        user = await get_ad_user({

            "query": issue
        })

        return await reset_permissions({

            "user": user
        })


    else:

        return await create_jira_ticket({

            "project": "IT",

            "priority": "Medium"
        })


# ======================================
# STEP 6: Retry Pattern (Resilience)
# ======================================
async def call_tool_with_retry(

    primary_tool,

    fallback_tool=None,

    max_retries=3
):

    delay = 1


    for attempt in range(max_retries):

        try:

            return await primary_tool({})


        except Exception as e:

            logging.warning(

                f"Attempt {attempt+1} failed: {e}"
            )


            if attempt == max_retries - 1:

                if fallback_tool:

                    logging.info(

                        "Switching to fallback tool"
                    )

                    return await fallback_tool({})

                raise


            await asyncio.sleep(delay)

            delay *= 2


# ======================================
# STEP 7: Run Demo
# ======================================
async def run_demo():

    print("\n=== Smart IT Triage Demo ===")


    result = await smart_it_triage(

        issue="VPN not working for employee",

        severity="high"
    )


    print("🔍 Routing Result:")

    print(result)


    print("\n=== Retry Pattern Demo ===")


    data = await call_tool_with_retry(

        query_postgres,

        fallback_tool=query_sqlite_cache
    )


    print("📊 Data Result:")

    print(data)


# ======================================
# STEP 8: Execute
# ======================================
await run_demo()


=== Smart IT Triage Demo ===


INFO:httpx:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


🧠 Classified as: network
🔍 Routing Result:
🎫 Jira ticket created: {'project': 'NET', 'priority': 'High'}

=== Retry Pattern Demo ===
📊 Data Result:
📦 Data from Postgres
